In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("AP001.csv")

In [3]:
df.head()

,From Date,To Date,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),SO2 (ug/m3),CO (mg/m3),...,Temp (degree C),RH (%),WS (m/s),WD (deg),SR (W/mt2),BP (mmHg),VWS (m/s),Xylene (ug/m3),RF (mm),AT (degree C)
0,2016-07-01 10:00:00,2016-07-01 11:00:00,10.67,39.0,17.67,39.2,32.33,7.07,6.60,0.48,...,33.43,71.67,2.30,226.33,123.67,NaN,-0.1,0.1,0.0,23.05
1,2016-07-01 11:00:00,2016-07-01 12:00:00,2.00,39.0,20.50,41.9,35.80,7.40,NaN,0.49,...,33.70,70.00,2.50,223.00,186.00,NaN,-0.1,0.1,0.0,NaN
2,2016-07-01 12:00:00,2016-07-01 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-07-01 13:00:00,2016-07-01 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-07-01 14:00:00,2016-07-01 15:00:00,20.50,50.0,15.40,43.6,32.78,6.35,6.38,0.47,...,33.57,63.50,1.88,223.00,240.50,NaN,-0.1,0.1,0.0,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59150 entries, 0 to 59149
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   From Date        59150 non-null  object 
 1   To Date          59150 non-null  object 
 2   PM2.5 (ug/m3)    54323 non-null  float64
 3   PM10 (ug/m3)     54450 non-null  float64
 4   NO (ug/m3)       55153 non-null  float64
 5   NO2 (ug/m3)      55100 non-null  float64
 6   NOx (ppb)        55315 non-null  float64
 7   NH3 (ug/m3)      53564 non-null  float64
 8   SO2 (ug/m3)      54285 non-null  float64
 9   CO (mg/m3)       54673 non-null  float64
 10  Ozone (ug/m3)    54567 non-null  float64
 11  Benzene (ug/m3)  55213 non-null  float64
 12  Toluene (ug/m3)  55213 non-null  float64
 13  Temp (degree C)  55113 non-null  float64
 14  RH (%)           55281 non-null  float64
 15  WS (m/s)         55299 non-null  float64
 16  WD (deg)         54656 non-null  float64
 17  SR (W/mt2)  

In [5]:
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.replace('(', '')
df.columns = df.columns.str.replace(')', '')
df.columns = df.columns.str.replace('/', '_')
df.columns = df.columns.str.replace('%', '')

In [6]:
df['From_Date'] = pd.to_datetime(df['From_Date'])
df = df.sort_values('From_Date')

In [7]:
df['AQI'] = df['PM2.5_ug_m3']
df = df.dropna(subset=['AQI'])

In [8]:
df = df.drop('BP_mmHg', axis=1)

df = df.fillna(method='ffill')
df = df.fillna(method='bfill')

/tmp/ipykernel_3212/2066805863.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')
/tmp/ipykernel_3212/2066805863.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill')


In [9]:
df['year'] = df['From_Date'].dt.year
df['month'] = df['From_Date'].dt.month
df['day'] = df['From_Date'].dt.day
df['dayofweek'] = df['From_Date'].dt.dayofweek

In [10]:
def season(m):
    if m in [12,1,2]: return 0
    if m in [3,4,5]: return 1
    if m in [6,7,8,9]: return 2
    return 3

df['season'] = df['month'].apply(season)

In [11]:
# lag features
lags = [1,2,3,7,14]

for lag in lags:
    df[f'lag_{lag}'] = df['AQI'].shift(lag)

df['roll7'] = df['AQI'].shift(1).rolling(7).mean()
df['roll14'] = df['AQI'].shift(1).rolling(14).mean()

df = df.dropna()

In [12]:
#train test split
split = int(len(df)*0.8)

train = df.iloc[:split]
test  = df.iloc[split:]

X_train = train.drop(['AQI','From_Date','To_Date'], axis=1)
y_train = train['AQI']

X_test = test.drop(['AQI','From_Date','To_Date'], axis=1)
y_test = test['AQI']

In [13]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
X_train = X_train.drop('PM2.5_ug_m3', axis=1)
X_test  = X_test.drop('PM2.5_ug_m3', axis=1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [14]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(((y_test - pred)**2).mean())
r2 = r2_score(y_test, pred)

print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))
print("R2:", round(r2,3))

MAE: 4.14
RMSE: 5.71
R2: 0.935


In [15]:
import joblib
joblib.dump(model, "aqi_model.pkl")

['aqi_model.pkl']